# Local Model Playground

Interactive notebook to pick a local VLM and run a single 2AFC trial on GPU.

In [ ]:
# Cell 1 — Setup & imports
import sys
from pathlib import Path

# Ensure the repo root is on sys.path so we can import evaluation_pipe
REPO_ROOT = str(Path.cwd().parent)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from dotenv import load_dotenv
load_dotenv()  # loads HF_TOKEN etc.

from PIL import Image
from IPython.display import display, HTML

from evaluation_pipe.models import create_model, list_models
from evaluation_pipe.models.base import ModelResponse

print("Imports OK")

In [ ]:
# Cell 2 — Choose model
print("Available models:", list_models())

# ---- Edit these to change model / device ----
MODEL_NAME = "smolvlm"
DEVICE = "cuda"

In [ ]:
# Cell 3 — Load model
model = create_model(MODEL_NAME, device=DEVICE)
print(f"Loaded model: {model.name}  (device={DEVICE})")

In [ ]:
# Cell 4 — Prepare trial images
#
# Default: 3 solid-color dummy images (reference=red, A=green, B=blue)
# Replace with Image.open("path/to/image.png") for real stimuli.

SIZE = (224, 224)

reference = Image.new("RGB", SIZE, color="red")
image_a   = Image.new("RGB", SIZE, color="green")
image_b   = Image.new("RGB", SIZE, color="blue")

# --- Uncomment below to load real images instead ---
# reference = Image.open("path/to/reference.png").convert("RGB")
# image_a   = Image.open("path/to/image_a.png").convert("RGB")
# image_b   = Image.open("path/to/image_b.png").convert("RGB")

# Display side-by-side
display(HTML("<b>Reference &nbsp; | &nbsp; Image A &nbsp; | &nbsp; Image B</b>"))
display(reference, image_a, image_b)

In [ ]:
# Cell 5 — Define prompt
PROMPT = (
    "You are given three images. The first image is the reference. "
    "Which of the other two images (A or B) is more similar to the reference? "
    "Answer with just 'A' or 'B'."
)

In [ ]:
# Cell 6 — Run single trial
response: ModelResponse = model.generate(
    images=[reference, image_a, image_b],
    prompt=PROMPT,
)

print(f"Raw text:        {response.raw_text}")
print(f"Generation time: {response.generation_time_s:.2f}s")
print(f"Model:           {response.model_name}")
print(f"Tokens:          {response.num_tokens_generated}")

In [ ]:
# Cell 7 — Cleanup
model.unload()
print("Model unloaded — GPU memory freed.")